# Supervised Wishart evaluation on real data

This notebook builds C-PolSARPro training inputs from the real San Francisco label map,
runs the legacy supervised Wishart classifier on the same ROI as the Python version,
and compares the two outputs.


In [ ]:
%load_ext autoreload
%autoreload 2

import os
import subprocess
from pathlib import Path

import dask.array as da
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from dask.diagnostics import ProgressBar
from scipy.ndimage import label as cc_label

from polsarpro.classification import (
    _label_training_clusters,
    _update_wishart_class_centers,
    wishart_supervised,
)
from polsarpro.dev.io import read_psp_bin
from polsarpro.io import open_netcdf_beam
from polsarpro.util import S_to_T3, boxcar

c_psp_root = Path("/home/c_psp/Soft")
c_classifier = c_psp_root / "bin" / "data_process_sngl" / "wishart_supervised_classifier.exe"

input_alos_data = Path("/data/psp/test_files/SAN_FRANCISCO_ALOS1_slc.nc")
input_test_dir = Path("/data/psp/SAN_FRANCISCO_ALOS1/T3")
label_file = Path("/polsarpro-dev/data/sample_labels_sf_alos1.nc")
mask_file = input_test_dir / "mask_valid_pixels.bin"

output_test_dir = Path("/data/psp/res/wishart_supervised_real_data")
output_test_dir.mkdir(parents=True, exist_ok=True)

boxcar_size = [7, 7]


def write_config_file(directory: Path, nrow: int, ncol: int) -> Path:
    """Create the minimal PSP config.txt needed by read_psp_bin."""
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    config_path = directory / "config.txt"
    config_path.write_text(
        f"""Nrow
{nrow}
---------
Ncol
{ncol}
---------
PolarCase
monostatic
---------
PolarType
full
"""
    )
    return config_path


def write_area_file(training_labels: np.ndarray, file_name: Path) -> dict[int, int]:
    """Write the text file consumed by create_class_map() in the C implementation."""
    file_name = Path(file_name)
    file_name.parent.mkdir(parents=True, exist_ok=True)

    # The legacy C parser ignores the explicit class value written in the file and
    # instead assigns semantic labels from the class-block position (1, 2, 3, ...).
    # To preserve original labels when a crop removes an intermediate class, we must
    # still emit empty blocks for missing class ids.
    class_values = list(range(1, int(training_labels.max()) + 1))
    lines = ["classes", str(len(class_values))]
    polygons_per_class = {}

    for class_value in class_values:
        regions, nregions = cc_label(training_labels == class_value)
        polygons_per_class[class_value] = int(nregions)
        lines.extend(["class", str(class_value), "areas", str(nregions)])

        for region_id in range(1, nregions + 1):
            coords = np.argwhere(regions == region_id)
            lines.extend(["area", str(region_id), str(len(coords))])
            for point_id, (yy, xx) in enumerate(coords, start=1):
                lines.extend(["pt", str(point_id), str(float(yy)), "x", str(float(xx))])

    file_name.write_text("\n".join(lines) + "\n")
    return polygons_per_class


def write_cluster_file(filtered_t3: xr.Dataset, training_labels: np.ndarray, file_name: Path):
    """Write the C cluster binary: ncluster as float32, then one T3 center per cluster."""
    file_name = Path(file_name)
    file_name.parent.mkdir(parents=True, exist_ok=True)

    lab, cluster_classes = _label_training_clusters(training_labels.astype(np.int32, copy=False))
    lab_da = da.from_array(lab, chunks=lab.shape)
    centers = _update_wishart_class_centers(filtered_t3, lab_da, nclass=len(cluster_classes) - 1).compute()

    rows = []
    for center in np.asarray(centers):
        rows.append(
            [
                np.real(center[0, 0]),
                np.real(center[0, 1]),
                np.imag(center[0, 1]),
                np.real(center[0, 2]),
                np.imag(center[0, 2]),
                np.real(center[1, 1]),
                np.real(center[1, 2]),
                np.imag(center[1, 2]),
                np.real(center[2, 2]),
            ]
        )

    with open(file_name, "wb") as f:
        np.array([len(rows)], dtype=np.float32).tofile(f)
        np.asarray(rows, dtype=np.float32).ravel().tofile(f)

    return lab, cluster_classes, np.asarray(rows, dtype=np.float32)


def summarize_agreement(reference: np.ndarray, candidate: np.ndarray, labels: np.ndarray):
    summary = {
        "overall_agreement": float((reference == candidate).mean()),
        "agreement_outside_training": float((reference[labels == 0] == candidate[labels == 0]).mean()),
        "agreement_on_training": float((reference[labels > 0] == candidate[labels > 0]).mean()),
    }

    classes = np.unique(np.concatenate([reference.ravel(), candidate.ravel()]))
    per_class = []
    for class_value in classes:
        mask = reference == class_value
        if mask.any():
            per_class.append((int(class_value), int(mask.sum()), float((candidate[mask] == class_value).mean())))

    return summary, per_class


## Prepare the image and the C training files

This notebook uses the full image by default. If we later need a faster turnaround,
we can uncomment the optional manual crop below and apply the same slice to the image
and the label map.

If a manual crop is used, small differences may appear near the crop borders. The
legacy C executable effectively averages with surrounding context from the original
scene, while the Python workflow here applies the boxcar on the already cropped data.
This is a user-side comparison detail rather than a critical algorithm issue, and we
do not change the internal Python workflow for it because that would complicate and
break the current usage pattern.

For the legacy comparison we intentionally point the C executable to the PolSARPro `T3`
directory and run it with `-iodf T3`. This avoids the ambiguous `S2` path, where the
legacy code switches between `T3` and `T4` depending on `PolarCase` in `config.txt`.


In [ ]:
labels_ds = xr.open_dataset(label_file)
labels_full = labels_ds["labels"].astype(np.int16)

row1, row2 = 0, labels_full.sizes["y"]
col1, col2 = 0, labels_full.sizes["x"]

# Optional manual crop for quicker iteration if needed later.
# row1, row2 = 8000, 12000
# col1, col2 = 200, 1100

labels_roi = labels_full.isel(y=slice(row1, row2), x=slice(col1, col2)).copy()
labels_np = labels_roi.values.astype(np.int16, copy=False)

print(f"ROI rows: {row1}:{row2} ({row2-row1})")
print(f"ROI cols: {col1}:{col2} ({col2-col1})")
print("Semantic classes:", np.unique(labels_np[labels_np > 0]).tolist())
print("Training pixels:", int((labels_np > 0).sum()))

S_roi = open_netcdf_beam(input_alos_data).isel(y=slice(row1, row2), x=slice(col1, col2))
T3_roi = boxcar(S_to_T3(S_roi), dim_az=boxcar_size[0], dim_rg=boxcar_size[1])

area_file = output_test_dir / "wishart_training_areas.txt"
cluster_file = output_test_dir / "wishart_training_clusters.bin"

polygons_per_class = write_area_file(labels_np, area_file)
lab, cluster_classes, cluster_centers = write_cluster_file(T3_roi, labels_np, cluster_file)

print("Polygons per class:", polygons_per_class)
print("Connected components:", len(cluster_classes) - 1)
print("Area file:", area_file)
print("Cluster file:", cluster_file)
print("First cluster center:")
cluster_centers[0]


## Run the C implementation

The `area` file only restores cluster-to-class mapping in the legacy code; the actual
cluster statistics come from the binary file we just generated from the Python-side T3 means.

The legacy executable also returns exit code `1` even when it completes successfully,
so the cell below accepts both `0` and `1` as non-error return codes.


In [ ]:
c_cmd = [
    str(c_classifier),
    "-id", str(input_test_dir),
    "-od", str(output_test_dir),
    "-iodf", "T3",
    "-af", str(area_file),
    "-nwr", str(boxcar_size[0]),
    "-nwc", str(boxcar_size[1]),
    "-ofr", str(row1),
    "-ofc", str(col1),
    "-fnr", str(row2 - row1),
    "-fnc", str(col2 - col1),
    "-cf", str(cluster_file),
    "-bmp", "0",
    "-errf", "/tmp/MemoryAllocError.txt",
    "-mask", str(mask_file),
]

print(" ".join(c_cmd))
completed = subprocess.run(c_cmd, check=False)
if completed.returncode not in (0, 1):
    raise RuntimeError(f"C classifier failed with exit code {completed.returncode}")
write_config_file(output_test_dir, nrow=row2 - row1, ncol=col2 - col1)

## Run the Python implementation on the same ROI


In [ ]:
out_file = output_test_dir / "wishart_supervised.nc"

# remove if file already exists as xarray will not overwrite existing files and will error instead
if out_file.exists():
    out_file.unlink()

with ProgressBar():
    out_py = wishart_supervised(
        S_roi,
        training_labels=labels_roi,
        boxcar_size=boxcar_size,
    ).to_netcdf(out_file)

out_py = xr.open_dataset(out_file) 
class_py = out_py["wishart_supervised_class"].values.astype(np.int16)
class_py


In [ ]:
print(out_file)

## Numerical comparison


In [ ]:
c_output_path = output_test_dir / f"wishart_supervised_class_{boxcar_size[0]}x{boxcar_size[1]}.bin"
class_c = read_psp_bin(c_output_path).astype(np.int16)
summary, per_class = summarize_agreement(class_c, class_py, labels_np)

print("Agreement summary")
for key, value in summary.items():
    print(f"- {key}: {value:.4f}")

print("\nAgreement by C class")
for class_value, npix, agreement in per_class:
    print(f"- class {class_value}: {npix} px, agreement={agreement:.4f}")

print("\nClass histograms")
for source_name, arr in [("C", class_c), ("Python", class_py)]:
    values, counts = np.unique(arr, return_counts=True)
    print(source_name, dict(zip(values.tolist(), counts.tolist())))


## Quick-look plots


In [ ]:
disagreement = np.where(class_c == class_py, 0, class_py)

fig, axes = plt.subplots(2, 2, figsize=(14, 12), constrained_layout=True)

l1 = axes[0, 0].imshow(labels_np[::8], cmap="tab10", interpolation="nearest", vmin=0, vmax = np.max(labels_np))
axes[0, 0].set_title("Training labels")
axes[0, 0].axis("off")
fig.colorbar(l1, ax=axes[0, 0], fraction=0.046)

l2 =axes[0, 1].imshow(class_c[::8], cmap="tab10", interpolation="nearest", vmin=0, vmax = np.max(labels_np))
axes[0, 1].set_title("C supervised Wishart")
axes[0, 1].axis("off")
fig.colorbar(l2, ax=axes[0, 1], fraction=0.046)

l3 = axes[1, 0].imshow(class_py[::8], cmap="tab10", interpolation="nearest", vmin=0, vmax = np.max(labels_np))
axes[1, 0].set_title("Python supervised Wishart")
axes[1, 0].axis("off")
fig.colorbar(l3, ax=axes[1, 0], fraction=0.046)

im = axes[1, 1].imshow(disagreement[::8], cmap="magma", interpolation="nearest")
axes[1, 1].set_title("Disagreement map (0 means match)")
axes[1, 1].axis("off")
fig.colorbar(im, ax=axes[1, 1], fraction=0.046)

plt.show()


Interpretation:

If agreement is high but not perfect, that is expected. The goal here is to verify that the
Python implementation lands in the same regime as the C executable on the real training setup,
not to force bit-identical output.
